### Install Qucircuit
Ensure you have qucircuit installed, the next cell has `pip install` for that.  
Also, take a look [here](https://github.com/atulvarshneya/quantum-computing/tree/master/examples/qckt) for several example algorithm implementations, as well as Getting Started tutorials on qckt.

# Problem Statement

Given a function $f(x)$ that takes 1 qubit as input, whose output is either CONSTANT (fixed output for both inputs) or BALANCED (output is equally 0 and 1 across both possible inputs).

Problem is to find which type it is.

# Imports

In [16]:
import qckt
from qckt.backend import Qdeb
import numpy as np
# import random as rnd

# Function to construct circuit for a CONSTANT or BALANCED function.

Argument fntype selects one of the 4 functions -

|`fntype`|$f(0)$|$f(1)$||
|---|---|---|---|
|0|0|0|CONSTANT. Always 0|
|1|0|1|BALANCED. f(0)=0, f(1)=1|
|2|1|0|BALANCED. f(0)=1, f(1)=0|
|3|1|1|CONSTANT. Always 1|


In [17]:
def get_fxckt(fntype):
	"""
	Returns a quantum circuit for the specified function type.
	IMPORTANT: MSB is the output qubit, LSB is the input qubit.
	"""
	fxckt = qckt.QCkt(2,2)
	match fntype:
		case 0:
			print("f(x) is CONSTANT. Always 0.")
			pass
		case 1:
			print("f(x) is BALANCED. f(0) = 0, f(1) = 1.")
			fxckt.CX(0, 1)
		case 2:
			print("f(x) is BALANCED. f(0) = 1, f(1) = 0.")
			fxckt.X(1)
			fxckt.CX(0, 1)
		case 3:
			print("f(x) is CONSTANT. Always 1.")
			fxckt.X(1)
		case _:
			raise ValueError("Invalid function type.")
	# fxckt.draw()
	print()
	return fxckt

Circuit size

In [18]:
# This is a 2-qubit circuit
nqubits = 2

Randomly select one type of the $f(x)$ functions, and create a custom operator `fx` as the oracle.

In [19]:
# randomly select a function type (0 to 3)
fntype = int(np.random.random() * 4.0)
print(f"Randomly selected function type: {fntype}")

# create the fx gate and add it to the qckt library
# note only QCkt() created after this will have access to this fx gate
fx_ckt = get_fxckt(fntype)
opMatrix = fx_ckt.to_opMatrix()
qckt.define_gate('fx', opMatrix=opMatrix)

Randomly selected function type: 3
f(x) is CONSTANT. Always 1.



# Construct the Deutsch-Josza circuit to determine the function type

Input, output qubits

In [20]:
# input qubit is inqubit, output is outqubit
inqubit = 0
outqubit = 1



Construct the Deutsche algorithm circuit

In [21]:
# create the Deutsch algorithmcircuit
deutsch_ckt = qckt.QCkt(nqubits=nqubits,nclbits=nqubits, name="Deutsch Algorithm")

# 1. prepare the output qubit in |-> state
deutsch_ckt.X(outqubit)
deutsch_ckt.H(outqubit)

# 2. apply H to the input qubit
deutsch_ckt.H(inqubit)

# 3. apply f(x) - append the fx circuit to the deutsch_ckt
deutsch_ckt.Border()
deutsch_ckt.fx(outqubit, inqubit)  ## note as the fx function is written, MSB is the output and LSB is the input
deutsch_ckt.Border()

# 4. apply H to the input qubit again
deutsch_ckt.H(inqubit)

# 5. measure the input qubit
deutsch_ckt.M([inqubit])

deutsch_ckt.draw()

Deutsch Algorithm
q000 ---------[H]--#-[fx L]--#-[H]-[M]-
                   # |    |  #      |  
q001 -[X]-[H]------#-[fx M]--#------|--
                   #         #      |  
creg ==============#=========#======v==
                   #         #         


# Run the circuit, and use the measured output to determine the result

In [ ]:
job = qckt.Job(deutsch_ckt,qtrace=False, shots=1)
bk = Qdeb()
bk.runjob(job)
res = job.get_creg()
print(f'Measured results: {res.intvalue}')  # outqubit is not measured, so it will be 0. So, effectively only inqubit value is returned.

if res.intvalue != 0:
	print("Found fx is BALANCED")
else:
	print("Found fx is CONSTANT")

Measured results: 0
Found fx is CONSTANT
